# Implementation of Ren-Zhao method and comparison
The sign of `e_i·e_j` follows the explicit twist function σ(A,B) of
Ren–Zhao (2023), cited below. `hypercomplex-engine` evaluates this sign law
in O(1) word operations (three trailing-zero counts, one maximum comparison,
one population count), with no dependence on the algebra's dimension. An
executable equivalence test against the published twist function
(`examples/ren_zhao_twist_check.ipynb`) verifies the claim on every basis
pair up to dimension 2^10 in both the standard and split families.

In [1]:
"""
verify_sign_equivalence.py
==========================
Three-way equivalence check for Cayley-Dickson basis multiplication signs:

  (1) REF   - raw Cayley-Dickson doubling formula, computed recursively on unit
              basis elements (independent of all block rules / bitwise logic).
  (2) RZ    - Ren & Zhao twist function, arXiv:2205.07986 (Thm 1, standard;
              Thm 2, split: sigma_s = sigma + a_{n-1}*b_{n-1}).
  (3) MBS   - O(1) bitwise evaluator of M. Ben Abdessalem (Sep 2026):
              three trailing-zero counts (nu), one max comparison, one
              popcount parity; split = standard core + top-level block-d rules.

Checks every basis pair (i, j) at dimensions n = 0..N_MAX for:
  standard algebra   :  MBS(i,j)   == RZ_sigma(i,j)   == REF(n,i,j)
  split algebra      :  MBS_split  == RZ_sigma_s     == REF(n,i,j,split)

If anything disagrees, the failing tuples are printed. Otherwise the run
prints a zero-mismatch report per dimension.
"""

from functools import lru_cache

N_MAX = 12   # 2^10 = 1024 basis elements; exhaustive = 1,398,101 pairs per mode

# ---------------------------------------------------------------- (1) REF
@lru_cache(maxsize=None)
def ref_unit(n, i, j, split_top=False):
    """(sign, index) of e_i * e_j in A_n via the raw doubling formula:
    (a,b)(c,d) = (a c - d* b, d a + b c*)   [standard]
    (a,b)(c,d) = (a c + d* b, d a + b c*)   [split_top, at the top level only]"""
    if n == 0:
        return (1, 0)
    h = 1 << (n - 1)
    si, ii = (0, i) if i < h else (1, i - h)
    sj, jj = (0, j) if j < h else (1, j - h)
    js = 1 if jj == 0 else -1                      # conjugation on a basis unit
    if si == 0 and sj == 0:
        return ref_unit(n - 1, ii, jj)
    if si == 0 and sj == 1:
        s, k = ref_unit(n - 1, jj, ii)
        return (s, k + h)
    if si == 1 and sj == 0:
        s, k = ref_unit(n - 1, ii, jj)
        return (js * s, k + h)
    s, k = ref_unit(n - 1, jj, ii)                 # e_j e_i ; anti-commute below
    inner = -s
    return ((js * inner) if not split_top else (-js * inner), k)

# ---------------------------------------------------------------- (2) RZ
def nu(x):
    return float("inf") if x == 0 else (x & -x).bit_length() - 1

def phi(a, b):
    return 0 if (a == 0 and b == 0) else 1

def rz_sigma(i, j, n):
    """Twist exponent of Ren-Zhao, Thm 1 (sign = (-1)^sigma)."""
    if i == 0 or j == 0:
        return 0
    if i == j:
        return 1
    di, dj, dx = nu(i), nu(j), nu(i ^ j)
    l = max(di, dj, dx)
    if di > dj:
        s = ((j >> l) & 1) + sum(phi((i >> b) & 1, (j >> b) & 1) for b in range(l, n))
    elif di < dj:
        s = 1 + ((i >> l) & 1) + sum(phi((i >> b) & 1, (j >> b) & 1) for b in range(l, n))
    else:
        s = ((i >> l) & 1) + sum(phi((i >> b) & 1, (j >> b) & 1) for b in range(l, n))
    return s % 2

def rz_sign(i, j, n):
    return 1 if rz_sigma(i, j, n) == 0 else -1

def rz_split_sign(i, j, n):
    """Ren-Zhao, Thm 2: sigma_s = sigma + a_{n-1} * b_{n-1} (top bits)."""
    s = rz_sigma(i, j, n)
    s ^= ((i >> (n - 1)) & 1) & ((j >> (n - 1)) & 1)
    return 1 if s == 0 else -1

# ---------------------------------------------------------------- (3) MBS
def pc(x):
    return x.bit_count()

def mbs_sign(i, j):
    """O(1) standard-algebra sign evaluator (dimension-independent)."""
    if i == 0 or j == 0:
        return 1
    if i == j:
        return -1
    t, ti, tj = nu(i ^ j), nu(i), nu(j)
    if t >= ti and t >= tj:
        k = t + 1
        s = 2 * ((i >> t) & 1) - 1
    elif ti >= tj:
        k = ti + 1
        b = (j >> (k - 1)) & 1
        j_loc = j & ((1 << (k - 1)) - 1)
        s = (-1 if b else 1) * (1 if j_loc == 0 else -1)
    else:
        k = tj + 1
        s = 1 - 2 * ((i >> tj) & 1)
    if pc((i | j) >> k) & 1:
        s = -s
    return s

def mbs_split_sign(i, j, dim):
    """Split algebra: split sign applied at the top doubling, standard base."""
    if i == 0 or j == 0:
        return 1
    half = 1 << (dim - 1)
    if i == j:
        return 1 if i >= half else -1
    if i >= half and j >= half:
        il, jl = i - half, j - half
        if il == jl:
            return 1
        if il == 0:
            return -1
        if jl == 0:
            return 1
        return mbs_sign(il, jl)
    return mbs_sign(i, j)

# ---------------------------------------------------------------- CHECKS
def main():
    bad = []
    for n in range(1, N_MAX + 1):
        for i in range(1 << n):
            for j in range(1 << n):
                ref_std = ref_unit(n, i, j)[0]
                if mbs_sign(i, j) != ref_std or rz_sign(i, j, n) != ref_std:
                    bad.append(("STD", n, i, j,
                                mbs_sign(i, j), rz_sign(i, j, n), ref_std))
                ref_spl = ref_unit(n, i, j, split_top=True)[0]
                if mbs_split_sign(i, j, n) != ref_spl or rz_split_sign(i, j, n) != ref_spl:
                    bad.append(("SPLIT", n, i, j,
                                mbs_split_sign(i, j, n), rz_split_sign(i, j, n), ref_spl))
        print(f"n={n:2d} checked ({(1 << n) ** 2:,} pairs x2 modes)")

    if bad:
        print(f"\nFAILURES: {len(bad)}")
        for row in bad[:20]:
            print(row)
        raise SystemExit(1)
    pairs = 2 * sum((1 << n) ** 2 for n in range(1, N_MAX + 1))
    print(f"\nOK: zero mismatches across {pairs:,} comparisons "
          f"(MBS == Ren-Zhao == raw doubling, standard and split, n <= {N_MAX}).")

if __name__ == "__main__":
    main()

n= 1 checked (4 pairs x2 modes)
n= 2 checked (16 pairs x2 modes)
n= 3 checked (64 pairs x2 modes)
n= 4 checked (256 pairs x2 modes)
n= 5 checked (1,024 pairs x2 modes)
n= 6 checked (4,096 pairs x2 modes)
n= 7 checked (16,384 pairs x2 modes)
n= 8 checked (65,536 pairs x2 modes)
n= 9 checked (262,144 pairs x2 modes)
n=10 checked (1,048,576 pairs x2 modes)
n=11 checked (4,194,304 pairs x2 modes)
n=12 checked (16,777,216 pairs x2 modes)

OK: zero mismatches across 44,739,240 comparisons (MBS == Ren-Zhao == raw doubling, standard and split, n <= 12).
